In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch # Added this line
if torch.cuda.is_available():
    print("GPU ready:", torch.cuda.get_device_name(0))
else:
    print("No GPU -- go to Runtime > Change runtime type > T4 GPU then restart")

GPU ready: Tesla T4


In [3]:
!apt-get update -qq
!apt-get install -y colmap

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libamd2 libatk-bridge2.0-0
  libatk1.0-0 libatk1.0-data libatspi2.0-0 libcamd2 libccolamd2 libceres2
  libcholmod3 libcolamd2 libcxsparse3 libdouble-conversion3 libevdev2
  libfreeimage3 libgflags2.2 libglew2.2 libgoogle-glog0v5 libgtk-3-0
  libgtk-3-bin libgtk-3-common libgudev-1.0-0 libilmbase25 libinput-bin
  libinput10 libjxr0 libmd4c0 libmetis5 libmtdev1 libopenexr25 libqt5core5a
  libqt5dbus5 libqt5gui5 libqt5network5 libqt5svg5 libqt5widgets5 libraw20
  librsvg2-common libspqr2 libsuitesparseconfig5 libwacom-bin libwacom-common
  libwacom9 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-render-util0
  l

In [1]:
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
!pip install nerfstudio==1.1.5

In [ ]:
!which ns-process-data
!ns-process-data --help

/usr/local/bin/ns-process-data
usage: /usr/local/bin/ns-process-data [-h] {images,video,polycam,metashape,realitycapture,record3d,odm,aria}

╭─ options ────────────────────────────────────────────────────────────────────╮
│ -h, --help          show this help message and exit                          │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ subcommands ────────────────────────────────────────────────────────────────╮
│ (required)                                                                   │
│   • images          Process images into a nerfstudio dataset.                │
│                                                                              │
│                                                                              │
│                     1. Scales images to a specified size.                    │
│                     2. Calculates the camera poses for each image using      │
│                     `COLMAP <https://colmap.git

In [2]:
import os
if not os.path.exists("/content/gaussian-splatting"):
    !git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive /content/gaussian-splatting
    print("Cloned.")
else:
    print("Already cloned.")


Cloning into '/content/gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 30.87 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

In [3]:
%cd /content/gaussian-splatting
!pip install plyfile tqdm -q
!pip install submodules/diff-gaussian-rasterization -q
!pip install submodules/simple-knn -q
print("Gaussian Splatting dependencies installed.")


/content/gaussian-splatting
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
Gaussian Splatting dependencies installed.


In [4]:
!pip install ninja -q
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch

  Cloning https://github.com/NVlabs/tiny-cuda-nn/ to /tmp/pip-req-build-6v4ton90
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/tiny-cuda-nn/ /tmp/pip-req-build-6v4ton90
  Resolved https://github.com/NVlabs/tiny-cuda-nn/ to commit 749dd70c5afc5a9dadb85e5652ed65d55e0ba187
  Running command git submodule update --init --recursive -q
  Preparing metadata (setup.py) ... done
  Created wheel for tinycudann: filename=tinycudann-2.0-cp312-cp312-linux_x86_64.whl size=16145641 sha256=e94da85be910d654465a0d019bcf8246b9dd8a52b174d9533f445c4a3aaf111a
  Stored in directory: /tmp/pip-ephem-wheel-cache-lw3p86pr/wheels/2f/e8/5a/6f5fba4370cba8f29cff0bb004adb2bfbe0a148555d848019a
Successfully built tinycudann


In [5]:
import tinycudann as tcnn
print("tcnn OK:", tcnn.__file__)

tcnn OK: /usr/local/lib/python3.12/dist-packages/tinycudann/__init__.py


In [1]:
# Kill EVERYTHING on port 8000
import subprocess
subprocess.run(['pkill', '-9', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
import time
time.sleep(2)

# ============================================================================
# Shilpa3D FastAPI Server with ngrok (FIXED)
# ============================================================================

import os
import json
import shutil
import subprocess
from pathlib import Path
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from typing import List

SESSIONS_DIR = "/content/drive/MyDrive/shilpa3D/sessions"
PIPELINE_SCRIPT = "/content/drive/MyDrive/shilpa3D/scripts/pipeline.py"
os.makedirs(SESSIONS_DIR, exist_ok=True)

app = FastAPI(title="Shilpa3D")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/jobs")
async def start_job(
    session_id: str = Form(...),
    statue_name: str = Form(...),
    method: str = Form(...),
    images: List[UploadFile] = File(...),
):
    if method not in ["nerf", "gaussian", "both"]:
        raise HTTPException(status_code=400, detail="invalid method")
    if len(images) < 5:
        raise HTTPException(status_code=400, detail="need 5+ images")

    session_dir = Path(SESSIONS_DIR) / session_id
    images_dir = session_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    for image in images:
        dest = images_dir / image.filename
        with open(dest, "wb") as f:
            shutil.copyfileobj(image.file, f)

    env = {**os.environ, "QT_QPA_PLATFORM": "offscreen", "SHILPA_SESSIONS_DIR": SESSIONS_DIR}
    log_path = session_dir / "pipeline_log.txt"
    with open(log_path, "w") as log_file:
        subprocess.Popen(
            ["python3", PIPELINE_SCRIPT, session_id, statue_name, method],
            env=env, stdout=log_file, stderr=subprocess.STDOUT,
        )

    return {"status": "started", "session_id": session_id}

@app.get("/jobs/{session_id}/status")
def get_status(session_id: str):
    status_path = Path(SESSIONS_DIR) / session_id / "status.json"
    if not status_path.exists():
        raise HTTPException(status_code=404, detail="not found")
    return json.load(open(status_path))

@app.get("/jobs/{session_id}/results")
def get_results(session_id: str):
    manifest_path = Path(SESSIONS_DIR) / session_id / "results" / "manifest.json"
    if not manifest_path.exists():
        raise HTTPException(status_code=404, detail="not ready")
    return json.load(open(manifest_path))

@app.get("/jobs/{session_id}/files/{file_path:path}")
def get_file(session_id: str, file_path: str):
    full_path = Path(SESSIONS_DIR) / session_id / "results" / file_path
    if not full_path.exists():
        raise HTTPException(status_code=404, detail="not found")
    return FileResponse(full_path)

# ============================================================================
# Start server + ngrok
# ============================================================================

import threading
import uvicorn
from pyngrok import ngrok

def run_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

print("\n" + "="*80)
print("Starting Shilpa3D FastAPI Server")
print("="*80 + "\n")

# Start server
server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

time.sleep(2)

# Set ngrok token
NGROK_AUTH_TOKEN = "3HgQKPVVAB6CT69bX4ys3bQ9exq_5LGNWi7mH1RCb6Gy9CNfF"  # <-- REPLACE WITH YOUR ACTUAL TOKEN

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print(f"✓ Ngrok auth token set\n")
except Exception as e:
    print(f"✗ Error setting ngrok token: {e}")
    exit(1)

# Create tunnel and extract URL correctly
try:
    tunnel = ngrok.connect(8000)
    # Extract just the URL string
    public_url = str(tunnel).split('"')[1] if '"' in str(tunnel) else str(tunnel)
    print(f"✓ Tunnel created!")
    print(f"  Public URL: {public_url}")
    print(f"\n" + "="*80)
    print(f"Copy this into your Node backend's .env:")
    print(f"  PIPELINE_URL={public_url}")
    print("="*80 + "\n")
except Exception as e:
    print(f"✗ Failed to create tunnel: {e}")
    exit(1)

# Test tunnel
time.sleep(2)
import requests
try:
    r = requests.get(f"{public_url}/health", timeout=5)
    if r.status_code == 200:
        print(f"✓ Tunnel is working! {r.json()}\n")
    else:
        print(f"✗ Tunnel returned {r.status_code}\n")
except Exception as e:
    print(f"✗ Tunnel test failed: {e}\n")

print(f"✓ Server running on {public_url}")


Starting Shilpa3D FastAPI Server

✓ Ngrok auth token set

✓ Tunnel created!
  Public URL: https://landmine-dawdler-hardship.ngrok-free.dev

Copy this into your Node backend's .env:
  PIPELINE_URL=https://landmine-dawdler-hardship.ngrok-free.dev

✓ Tunnel is working! {'status': 'ok'}

✓ Server running on https://landmine-dawdler-hardship.ngrok-free.dev


In [7]:
import requests

# The URL should look like: https://abc-123-def.ngrok-free.dev
NGROK_URL = "https://landmine-dawdler-hardship.ngrok-free.dev"

r = requests.get(f"{NGROK_URL}/health")
print(f"Status: {r.status_code}")
if r.status_code == 200:
    print(f"Response: {r.json()}")
    print(f"\n✓ ngrok tunnel works!")
else:
    print(f"Still issues")

Status: 200
Response: {'status': 'ok'}

✓ ngrok tunnel works!


In [8]:
import os
import json

sessions_dir = "/content/drive/MyDrive/shilpa3D/sessions"

# List all sessions
sessions = os.listdir(sessions_dir)
print(f"Found {len(sessions)} sessions:\n")

for session_id in sorted(sessions):
    session_path = os.path.join(sessions_dir, session_id)

    # Check if it has a status file
    status_file = os.path.join(session_path, "status.json")
    if os.path.exists(status_file):
        with open(status_file) as f:
            status = json.load(f)
        print(f"✓ {session_id}")
        print(f"  Status: {status['stage']} {status['percent']}%")
    else:
        print(f"• {session_id} (no status yet)")

    print()

Found 6 sessions:

✓ 858680df-a60c-4151-8c7e-e01f65086eb0
  Status: done 100%

✓ smoketest01
  Status: done 100%

✓ smoketest02
  Status: done 100%

✓ test01
  Status: done 100%

✓ test02
  Status: done 100%

✓ test_no_bg
  Status: gaussian_render 90%



In [ ]:
import json
import time

session_id = "858680df-a60c-4151-8c7e-e01f65086eb0"
status_path = f"/content/drive/MyDrive/shilpa3D/sessions/{session_id}/status.json"

print("Monitoring pipeline...")
while True:
    with open(status_path) as f:
        status = json.load(f)

    print(f"[{time.strftime('%H:%M:%S')}] {status['stage']} {status['percent']}%")

    if status['percent'] == 100:
        print("\n✓ Done!")
        break

    time.sleep(30)  # Check every 30 seconds

Monitoring pipeline...
[16:19:50] colmap_features 10%
[16:20:20] colmap_features 10%
[16:20:50] colmap_features 10%
[16:21:20] colmap_features 10%
[16:21:50] colmap_features 10%
[16:22:20] colmap_features 10%
[16:22:50] colmap_features 10%
[16:23:20] colmap_features 10%
[16:23:50] colmap_features 10%
[16:24:20] colmap_features 10%
[16:24:50] colmap_features 10%
[16:25:20] colmap_matching 25%
[16:25:50] colmap_matching 25%
[16:26:20] colmap_matching 25%
[16:26:50] colmap_matching 25%
[16:27:20] colmap_matching 25%
[16:27:50] colmap_matching 25%
[16:28:20] colmap_matching 25%
[16:28:50] colmap_matching 25%
[16:29:20] colmap_matching 25%
[16:29:50] colmap_mapper 40%
[16:30:20] colmap_mapper 40%
[16:30:50] colmap_mapper 40%
[16:31:20] colmap_mapper 40%
[16:31:50] colmap_mapper 40%
[16:32:20] colmap_mapper 40%
[16:32:50] colmap_mapper 40%
[16:33:20] colmap_mapper 40%
[16:33:50] colmap_mapper 40%
[16:34:20] colmap_mapper 40%
[16:34:50] gaussian_training 80%
[16:35:20] gaussian_training 80%
[

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 90, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors

[17:16:50] done 100%

✓ Done!


In [ ]:
import cv2
import glob
import os

BASE = "/content/drive/MyDrive/shilpa3D"
SESSION_ID = "858680df-a60c-4151-8c7e-e01f65086eb0"

print("Creating final video deliverables...\n")

# Video 2: Gaussian Splatting
gs_renders = sorted(glob.glob(f"{BASE}/sessions/{SESSION_ID}/gaussian_out/train/ours_15000/renders/*.png"))
if gs_renders and len(gs_renders) > 0:
    first = cv2.imread(gs_renders[0])
    h, w = first.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(f"{BASE}/gaussian_sculpture_360.mp4", fourcc, 30.0, (w, h))

    for path in gs_renders:
        frame = cv2.imread(path)
        out.write(frame)
    out.release()
    size = os.path.getsize(f"{BASE}/gaussian_sculpture_360.mp4") / (1024*1024)
    print(f"✓ gaussian_sculpture_360.mp4 ({len(gs_renders)} frames, {size:.1f} MB)")

Creating final video deliverables...

✓ gaussian_sculpture_360.mp4 (17 frames, 2.2 MB)
